# Breakout Room #4 — Qini y comparación de algoritmos T-learner

**Objetivo:** construir la curva Qini y la tabla de deciles para comparar dos formas de
ordenar a los 14,000 clientes del experimento — por **riesgo/churn** y por **uplift
(T-learner)** — y ver cuál de las dos sirve de verdad para decidir a quién contactar.

Esta vez separamos los datos en dos partes: **entrenamiento** (para que los modelos
aprendan) y **prueba** (para evaluarlos con datos que nunca vieron — la misma idea de "las
dos ventanas" del Breakout 2, pero aplicada a modelos en vez de a fechas).

Donde dice **`# TODO`** hay algo para que ustedes completen.

In [0]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

# Databricks/Jupyter proporcionan display; el respaldo permite ejecutar como script.
try:
    display
except NameError:
    def display(objeto):
        print(objeto)

RUTA_DATOS = Path('/Volumes/workspace/default/semana3/uplift_experiment.csv')
RUTA_SALIDAS = RUTA_DATOS.parent
RANDOM_STATE = 42
FEATURES = ['recency_days', 'frequency', 'monetary', 'email_open_rate']

df = pd.read_csv(RUTA_DATOS)
requeridas = FEATURES + ['treatment_group', 'responded_90d']
faltantes = sorted(set(requeridas) - set(df.columns))

train, test = train_test_split(
    df, test_size=0.30, random_state=RANDOM_STATE,
    stratify=df['treatment_group']
)
for grupo in ['treatment', 'control']:
    if train.loc[train['treatment_group'] == grupo, 'responded_90d'].nunique() != 2:
        raise ValueError(f'El entrenamiento del grupo {grupo} necesita respuestas 0 y 1.')
if len(test) < 10:
    raise ValueError('Se requieren al menos diez clientes de prueba para construir deciles.')
print(f'Entrenamiento: {len(train):,} clientes · Prueba: {len(test):,} clientes')

## Paso 1 — entrenen los dos modelos (solo con datos de ENTRENAMIENTO)

- **`modelo_riesgo`**: un modelo de churn normal y corriente. No sabe nada de tratamiento/control
  — mezcla a todos los clientes de entrenamiento y aprende a predecir quién NO va a responder
  (`churned = 1 - responded_90d`). Así funcionaría un modelo de churn tradicional, el mismo tipo
  que construyeron en el Breakout 2.
- **`modelo_tratado`** y **`modelo_control`**: el T-learner de siempre (Breakout 3), entrenado
  solo con las filas de entrenamiento.


In [0]:
# --- SOLUCIÓN: modelo de riesgo/churn (no sabe de tratamiento) ---
train_riesgo = train.copy()
train_riesgo["churned"] = 1 - train_riesgo["responded_90d"]

modelo_riesgo = LogisticRegression(max_iter=1000)
modelo_riesgo.fit(train_riesgo[FEATURES], train_riesgo["churned"])

# T-learner: un modelo por grupo, SOLO con train
train_t = train[train["treatment_group"] == "treatment"]
train_c = train[train["treatment_group"] == "control"]

modelo_tratado = LogisticRegression(max_iter=1000)
modelo_control = LogisticRegression(max_iter=1000)
modelo_tratado.fit(train_t[FEATURES], train_t["responded_90d"])
modelo_control.fit(train_c[FEATURES], train_c["responded_90d"])

print("modelo_riesgo, modelo_tratado y modelo_control entrenados")


## Paso 2 — puntúen el conjunto de PRUEBA (el que ningún modelo vio)

Igual que en el Breakout 3: le preguntan a los modelos sobre todos los clientes de prueba,
tratados y de control por igual.


In [0]:
test = test.copy()
X_test = test[FEATURES]

# --- SOLUCIÓN: puntuamos el conjunto de prueba (que ningún modelo vio) ---
test["prob_churn"] = modelo_riesgo.predict_proba(X_test)[:, 1]
test["p_tratado"] = modelo_tratado.predict_proba(X_test)[:, 1]
test["p_control"] = modelo_control.predict_proba(X_test)[:, 1]
test["uplift_estimado"] = test["p_tratado"] - test["p_control"]

test[["prob_churn", "p_tratado", "p_control", "uplift_estimado"]].describe().round(3)


## Paso 3 — Curva Qini corregida

Para cada corte de la lista ordenada se calcula:

`ganancia(N) = (tasa de tratados en top N − tasa de control en top N) × N`

La diagonal del azar conecta el origen con la ganancia de contactar al 100%.
El coeficiente es el área entre curva y diagonal, dividida por el tamaño de prueba.
Se usan 40 cortes para todos los modelos, como en el ejercicio original.

Los cortes iniciales sin observaciones de ambos grupos se omiten: no se puede estimar
su diferencia de tasas. Los empates se resuelven con un orden aleatorio reproducible,
independiente de la respuesta y del tratamiento. Este mismo orden se usa en los deciles.
El valor depende de la definición y normalización de este ejercicio; al compararlo con
otra implementación de Qini, hay que verificar que use la misma fórmula.

In [0]:
def ordenar_por_score(datos, score_col):
    if not np.isfinite(datos[score_col].to_numpy(dtype=float)).all():
        raise ValueError(f'El score {score_col} contiene valores no finitos.')
    return (
        datos.sample(frac=1, random_state=RANDOM_STATE)
        .sort_values(score_col, ascending=False, kind='mergesort')
        .reset_index(drop=True)
    )


def curva_qini(test_df, score_col, steps=40):
    orden = ordenar_por_score(test_df, score_col)
    es_tratado = orden['treatment_group'].eq('treatment').to_numpy()
    respondio = orden['responded_90d'].to_numpy()
    acumulado_tratados = np.cumsum(np.where(es_tratado, respondio, 0))
    acumulado_control = np.cumsum(np.where(~es_tratado, respondio, 0))
    n_tratados_acum = np.cumsum(es_tratado)
    n_control_acum = np.cumsum(~es_tratado)
    n = len(orden)
    xs, ys = [0.0], [0.0]
    for cantidad in np.unique(np.linspace(1, n, steps).astype(int)):
        nt, nc = n_tratados_acum[cantidad - 1], n_control_acum[cantidad - 1]
        if nt == 0 or nc == 0:
            continue
        tasa_t = acumulado_tratados[cantidad - 1] / nt
        tasa_c = acumulado_control[cantidad - 1] / nc
        xs.append(cantidad / n)
        ys.append((tasa_t - tasa_c) * cantidad)
    if len(xs) < 2 or xs[-1] != 1.0:
        raise ValueError('La evaluación requiere clientes tratados y de control.')
    return np.array(xs), np.array(ys)


try:
    _trap = np.trapezoid
except AttributeError:
    _trap = np.trapz


def coeficiente_qini(xs, ys, n_clientes):
    diagonal_azar = xs * ys[-1]
    return float(_trap(ys - diagonal_azar, xs) / n_clientes)


xs_uplift, ys_uplift = curva_qini(test, 'uplift_estimado')
xs_riesgo, ys_riesgo = curva_qini(test, 'prob_churn')
qini_uplift = coeficiente_qini(xs_uplift, ys_uplift, len(test))
qini_riesgo = coeficiente_qini(xs_riesgo, ys_riesgo, len(test))
print(f'Qini corregido · uplift (logística): {qini_uplift:+.3f}')
print(f'Qini corregido · riesgo/churn:       {qini_riesgo:+.3f}')

In [0]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))
ax.plot(xs_uplift, ys_uplift, color="#00704A", linewidth=3, label=f"Modelo de uplift (Qini={qini_uplift:+.3f})")
ax.plot(xs_riesgo, ys_riesgo, color="#C4512F", linewidth=3, label=f"Modelo de riesgo/churn (Qini={qini_riesgo:+.3f})")
ax.plot([0, 1], [0, ys_uplift[-1]], color="#9A988E", linestyle="--", label="Al azar")
ax.set_xlabel("% de la base contactada (de prueba)")
ax.set_ylabel("Respuestas incrementales acumuladas")
ax.set_title("La curva Qini del experimento")
ax.legend()
plt.tight_layout()
plt.savefig(RUTA_SALIDAS / "curva_qini_corregida.png", dpi=150)
plt.show()

## Paso 4 — la tabla de deciles: ¿el modelo ordena de verdad?

Dividan el conjunto de prueba en 10 deciles según `uplift_estimado` (decil 1 = uplift más alto
predicho). Para cada decil, comparen el uplift PREDICHO promedio contra el uplift REAL (tasa de
respuesta de tratados menos tasa de respuesta de control, DENTRO de ese decil).


In [0]:
def construir_tabla_deciles(datos, score_col):
    orden = ordenar_por_score(datos, score_col)
    filas = []
    for decil, posiciones in enumerate(np.array_split(np.arange(len(orden)), 10), start=1):
        grupo = orden.iloc[posiciones]
        tratados = grupo.loc[grupo['treatment_group'] == 'treatment', 'responded_90d']
        controles = grupo.loc[grupo['treatment_group'] == 'control', 'responded_90d']
        tasa_t, tasa_c = tratados.mean(), controles.mean()
        filas.append({
            'decil': decil,
            'uplift_predicho': grupo[score_col].mean(),
            'uplift_real': tasa_t - tasa_c,
            'n': len(grupo), 'n_tratados': len(tratados), 'n_control': len(controles),
            'tasa_tratados': tasa_t, 'tasa_control': tasa_c,
        })
    return pd.DataFrame(filas)


def correlacion_deciles(tabla):
    validos = tabla[['uplift_predicho', 'uplift_real']].dropna()
    if len(validos) < 2 or (validos.nunique() < 2).any():
        return float('nan')
    return validos['uplift_predicho'].corr(validos['uplift_real'])


tabla_deciles = construir_tabla_deciles(test, 'uplift_estimado')
display(tabla_deciles.round(3))
print(f'Correlación por deciles · logística: {correlacion_deciles(tabla_deciles):.3f}')

## Para la clase — interpretar las salidas de esta ejecución

**1. ¿El coeficiente Qini del modelo de riesgo/churn es mayor, menor o parecido al del modelo de uplift? ¿Qué dice eso sobre usar un modelo de churn para decidir a quién contactar?**

 Mucho menor: uplift = +0.222 vs riesgo/churn = +0.006 (casi igual que el azar). Un modelo de churn NO sirve para decidir a quién contactar: marca a quien se va (Lost Cause), no a quien responde al incentivo. Para targeting hay que usar uplift.

**2. Miren la tabla de deciles: ¿hay algún decil donde lo predicho y lo real se alejan mucho? Con ~420 clientes por decil, ¿qué tan confiables son esos números?**

   El decil 6 es el que más se aleja (predijo +0.099, real -0.059, diferencia 0.158); el decil 1 también (0.394 vs 0.505). Con 420 por decil (~210 tratados y ~210 control) el error estándar es ≈ ±0.05, así que los deciles del medio son ruidosos. Confiable: la tendencia general (top claramente positivo, cola cerca de cero); NO confiable: el número exacto de un decil aislado.

**3. Si tuvieran que explicarle a alguien de negocio qué es el coeficiente Qini en una frase, ¿qué dirían?**

   Es cuánta ganancia extra sobre elegir al azar consigues al contactar a los primeros de tu lista: 0 = tu orden no sirve más que el azar; más alto = mejor concentras a los clientes que el cupón de verdad convence.


## Paso 5 — Comparar cuatro algoritmos dentro del T-learner

Cada algoritmo entrena dos clasificadores: uno para tratamiento y otro para control.
Se conservan `FEATURES`, `train_t`, `train_c` y `test`. La logística reutiliza los modelos
del Paso 1 para garantizar que su fila represente exactamente el modelo original.

Los árboles y Random Forest se entrenan únicamente con entrenamiento. No se ajustan
sus parámetros usando prueba. Random Forest usa 300 árboles y `max_depth=5` como
configuración fija inicial, no como configuración óptima demostrada.

La correlación se calcula con valores sin redondear. Los árboles pueden producir pocos
scores diferentes: los deciles se forman por posiciones, con empates aleatorios
reproducibles. Una correlación no definida se deja como `NaN`, nunca como cero.

In [0]:
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

modelos_base = {
    'Logística (la del notebook)': LogisticRegression(max_iter=1000),
    'Árbol sin límite': DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Árbol con max_depth=2': DecisionTreeClassifier(max_depth=2, random_state=RANDOM_STATE),
    'Random Forest (300 árboles, max_depth=5)': RandomForestClassifier(
        n_estimators=300, max_depth=5, random_state=RANDOM_STATE, n_jobs=-1
    ),
}

def probabilidad_respuesta(modelo, datos):
    indice_clase_1 = list(modelo.classes_).index(1)
    return modelo.predict_proba(datos)[:, indice_clase_1]

modelos_entrenados = {}
predicciones_por_modelo = {}
deciles_por_modelo = {}
curvas_por_modelo = {}
resultados_modelos = []

for nombre, modelo_base in modelos_base.items():
    if nombre == 'Logística (la del notebook)':
        modelo_t, modelo_c = modelo_tratado, modelo_control
    else:
        modelo_t, modelo_c = clone(modelo_base), clone(modelo_base)
        modelo_t.fit(train_t[FEATURES], train_t['responded_90d'])
        modelo_c.fit(train_c[FEATURES], train_c['responded_90d'])

    evaluacion = test.copy()
    evaluacion['p_tratado_modelo'] = probabilidad_respuesta(modelo_t, evaluacion[FEATURES])
    evaluacion['p_control_modelo'] = probabilidad_respuesta(modelo_c, evaluacion[FEATURES])
    evaluacion['uplift_modelo'] = evaluacion['p_tratado_modelo'] - evaluacion['p_control_modelo']
    deciles = construir_tabla_deciles(evaluacion, 'uplift_modelo')
    xs, ys = curva_qini(evaluacion, 'uplift_modelo')
    qini = coeficiente_qini(xs, ys, len(evaluacion))

    modelos_entrenados[nombre] = {'tratado': modelo_t, 'control': modelo_c}
    predicciones_por_modelo[nombre] = evaluacion
    deciles_por_modelo[nombre] = deciles
    curvas_por_modelo[nombre] = (xs, ys)
    resultados_modelos.append({
        'Modelo dentro del T-learner': nombre,
        'Correlación con el uplift real': correlacion_deciles(deciles),
        'Qini': qini,
    })

tabla_modelos = pd.DataFrame(resultados_modelos)
display(tabla_modelos.round(3))
print('Correlación: Pearson entre deciles. Qini: fórmula corregida del Paso 3.')

## Paso 6 — Curvas y evidencia por deciles

La tabla anterior es la comparación solicitada. Estas gráficas permiten revisar si un
Qini alto coincide con una curva que concentra ganancia al inicio. Se incluye riesgo/churn
como referencia adicional, fuera de las cuatro filas de algoritmos T-learner.

In [0]:
fig, ax = plt.subplots(figsize=(10, 6))
for nombre, (xs, ys) in curvas_por_modelo.items():
    valor_qini = tabla_modelos.loc[
        tabla_modelos['Modelo dentro del T-learner'] == nombre, 'Qini'
    ].iloc[0]
    ax.plot(xs, ys, linewidth=2, label=f'{nombre} (Qini={valor_qini:+.3f})')
ax.plot(xs_riesgo, ys_riesgo, ':', color='gray', label=f'Riesgo/churn ({qini_riesgo:+.3f})')
ax.plot([0, 1], [0, ys_uplift[-1]], '--', color='black', label='Al azar')
ax.set(xlabel='Fracción de la base de prueba contactada',
       ylabel='Respuestas incrementales acumuladas (estimadas)',
       title='Comparación T-learner · Qini corregido')
ax.legend(fontsize=8)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(RUTA_SALIDAS / 'comparacion_qini_modelos.png', dpi=150)
plt.show()

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True, sharey=True)
for ax, (nombre, tabla) in zip(axes.flat, deciles_por_modelo.items()):
    ax.plot(tabla['decil'], tabla['uplift_predicho'], 'o-', label='Predicho')
    ax.plot(tabla['decil'], tabla['uplift_real'], 's--', label='Observado')
    ax.axhline(0, color='gray', linewidth=0.8)
    ax.set(title=nombre, xlabel='Decil (1 = mayor uplift)', ylabel='Uplift')
    ax.set_xticks(range(1, 11))
    ax.grid(alpha=0.2)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(RUTA_SALIDAS / 'comparacion_deciles_modelos.png', dpi=150)
plt.show()

tabla_deciles_modelos = pd.concat([
    tabla.assign(modelo=nombre) for nombre, tabla in deciles_por_modelo.items()
], ignore_index=True)
display(tabla_deciles_modelos.round(3))

## Paso 7 — Revisar el 30% prioritario y guardar las tablas

El 30% es una política inicial a evaluar, no un porcentaje óptimo probado por Qini.
Se compara, para cada algoritmo, el promedio predicho del top 30% con la diferencia
observada entre tratados y control **en todo ese segmento**. También se muestran los
tamaños de muestra para interpretar la evidencia. No se calculan ganancias monetarias
sin incorporar margen incremental, costo de contacto y costo del incentivo.

El mayor Qini en prueba es descriptivo de esta comparación. Si se utiliza para elegir
modelo, se necesita validación independiente antes de afirmar su desempeño futuro.

In [0]:
filas_top30 = []
for nombre, evaluacion in predicciones_por_modelo.items():
    orden = ordenar_por_score(evaluacion, 'uplift_modelo')
    cantidad = max(1, int(np.ceil(0.30 * len(orden))))
    segmento = orden.iloc[:cantidad]
    tratados = segmento.loc[segmento['treatment_group'] == 'treatment', 'responded_90d']
    controles = segmento.loc[segmento['treatment_group'] == 'control', 'responded_90d']
    filas_top30.append({
        'Modelo': nombre, 'n_top30': len(segmento),
        'n_tratados': len(tratados), 'n_control': len(controles),
        'uplift_predicho_top30': segmento['uplift_modelo'].mean(),
        'uplift_observado_top30': tratados.mean() - controles.mean(),
    })
tabla_top30 = pd.DataFrame(filas_top30)
display(tabla_top30.round(3))

tabla_modelos.to_csv(RUTA_SALIDAS / 'comparacion_modelos_tlearner.csv', index=False)
tabla_deciles_modelos.to_csv(RUTA_SALIDAS / 'deciles_por_modelo.csv', index=False)
tabla_top30.to_csv(RUTA_SALIDAS / 'comparacion_top30_modelos.csv', index=False)
print(f'Tablas y gráficas guardadas en: {RUTA_SALIDAS}')